# Standalone leakage safety: build, train, inspect

This notebook builds a real `SkyulfPipeline` using synthetic **pandas** data.
No backend, Celery, canvas, downloads, or dataset files are needed.

Run the cells from top to bottom. Output tables and training metrics are computed
when you run them; the notebook does not contain invented execution results.

If the core library is not installed, run `uv pip install -e ./skyulf-core`
from the repository root. Select a Jupyter kernel from that same environment.

We will:
1. Build a linear pipeline similar to a canvas workflow.
2. Train it, inspect preprocessing, and make predictions.
3. Compare `raise`, `warn`, and `ignore` on an intentionally leaking order.
4. Supply an external `SplitDataset` and check train-only scaler statistics.
5. Refit a fresh pipeline inside each cross-validation fold.
6. Explain why backend `unsupported_graph` is different from the leakage gate.

In [1]:
import logging
from copy import deepcopy

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

from skyulf import SkyulfPipeline, validate_leakage_safety
from skyulf.data.dataset import SplitDataset
from skyulf.leakage import OnLeakage


def _require(condition: bool, message: str) -> None:
    """Keep tutorial correctness checks active under optimized Python."""
    if not condition:
        raise RuntimeError(message)


logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

## 1. Create raw data

The target depends on a numeric feature and a category. `unused_id` is a column
we deliberately exclude; dropping it does not require learning from the data.
We never use the target to generate an input feature.

In [2]:
rng = np.random.default_rng(42)
data = pd.DataFrame(
    {
        "x": np.linspace(1.0, 30.0, 60),
        "segment": np.resize(["A", "B", "C"], 60),
        "unused_id": np.arange(60),
    }
)
category_effect = data["segment"].map({"A": 0.0, "B": 5.0, "C": -3.0})
data["target"] = 3.0 * data["x"] + category_effect + rng.normal(0.0, 0.2, len(data))
display(data.head())

,x,segment,unused_id,target
0,1.000000,A,0,3.060943
1,1.491525,B,1,9.266579
2,1.983051,C,2,3.099243
3,2.474576,A,3,7.611842
4,2.966102,B,4,13.508098


## 2. Build a safe, linear pipeline

Here `FeatureGenerationNode` performs **row-local multiplication** (`x * 2`).
`DropMissingColumns` removes an explicit column with threshold zero. Neither
operation learns a dataset-wide statistic, so these configurations may precede
the split. The category encoder and scaler must follow it.

`feature_target_split` separates X from y. It does **not** create a train/test
boundary; that is the preceding `TrainTestSplitter`'s job.

Important: do not generalize this arithmetic example to every FeatureGeneration
operation. `group_agg` computes aggregates from its input batch during apply;
a `learns_from_data=False` fit flag alone does not prove that such an operation
is row-local or safe for held-out evaluation. This notebook does not use it.

In [3]:
feature_step = {
    "name": "double_x",
    "transformer": "FeatureGenerationNode",
    "params": {
        "operations": [
            {
                "operation_type": "arithmetic",
                "method": "multiply",
                "input_columns": ["x"],
                "constants": [2.0],
                "output_column": "x_double",
            }
        ],
    },
}
drop_step = {
    "name": "drop_unused_id",
    "transformer": "DropMissingColumns",
    "params": {"columns": ["unused_id"], "missing_threshold": 0},
}
split_step = {
    "name": "train_test_split",
    "transformer": "TrainTestSplitter",
    "params": {"test_size": 0.25, "random_state": 42, "target_column": "target"},
}
target_step = {
    "name": "separate_X_y",
    "transformer": "feature_target_split",
    "params": {"target_column": "target"},
}
encoding_step = {
    "name": "encode_segment",
    "transformer": "OneHotEncoder",
    "params": {"columns": ["segment"]},
}
scale_step = {
    "name": "scale_numeric",
    "transformer": "StandardScaler",
    "params": {"columns": ["x", "x_double"]},
}
safe_config = {
    "preprocessing": [
        feature_step,
        drop_step,
        split_step,
        target_step,
        encoding_step,
        scale_step,
    ],
    "modeling": {"type": "linear_regression"},
}

# An ordinary variable holding a SkyulfPipeline instance.
pipeline = SkyulfPipeline(deepcopy(safe_config))
_require(pipeline.validate_leakage_safety() == [], "The split-first pipeline must pass validation")

steps = (
    ["Dataset"]
    + [step["transformer"] for step in safe_config["preprocessing"]]
    + ["LinearRegression"]
)
display(pd.DataFrame({"step": range(len(steps)), "node": steps}))

,step,node
0,0,Dataset
1,1,FeatureGenerationNode
2,2,DropMissingColumns
3,3,TrainTestSplitter
4,4,feature_target_split
5,5,OneHotEncoder
6,6,StandardScaler
7,7,LinearRegression


## 3. Fit, inspect, and predict

The default `on_leakage="raise"` remains enabled. The model is fitted on the
training partition; `fit_metrics` contains the pipeline's evaluation results.

For prediction, pass **raw features without the target**. The pipeline applies
its already-fitted preprocessing itself.

In [4]:
fit_metrics = pipeline.fit(data, target_column="target")
_require(pipeline.is_fitted(), "The safe pipeline must fit successfully")

step_report = pd.DataFrame(fit_metrics["preprocessing"]["steps"].values())
display(step_report[["name", "transformer", "rows_in", "rows_out"]])
display(fit_metrics.get("modeling", {}))

new_rows = pd.DataFrame(
    {
        "x": [31.0, 32.0, 33.0],
        "segment": ["A", "B", "C"],
        "unused_id": [101, 102, 103],
    }
)
predictions = np.asarray(pipeline.predict(new_rows)).reshape(-1)
display(new_rows.assign(prediction=predictions))

,name,transformer,rows_in,rows_out
0,double_x,FeatureGenerationNode,60,60
1,drop_unused_id,DropMissingColumns,60,60
2,train_test_split,TrainTestSplitter,60,60
3,separate_X_y,feature_target_split,60,60
4,encode_segment,OneHotEncoder,60,60
5,scale_numeric,StandardScaler,60,60


{'problem_type': 'regression',
 'splits': {'train': ModelEvaluationReport(dataset_name='train', metrics={'mae': 0.12304951849731509, 'mse': 0.022222610998726827, 'rmse': 0.14907250249032122, 'r2': 0.9999618222028821, 'mape': 0.0048722690015778605, 'explained_variance': 0.9999618222028821}, classification=None, regression=RegressionEvaluation(residuals=ResidualsData(predicted=[25.084953626942728, 11.805462431495862, 11.853122887969413, 66.97522647764039, 13.85726169585294, 71.40172354278934, 35.989747021597715, 58.12223234734248, 93.53420886853411, 44.84274115189562, 82.62941547387913, 25.13261408341627, 42.83860234401209, 16.279619953118363, 47.265099409161046, 38.364444822389586, 31.56324995644876, 38.41210527886313, 89.10771180338517, 16.231959496644823, 47.2174389526875, 78.20291840873018, 60.49693014813436, 62.54872941249145, 42.790941887538544, 64.92342721328332, 9.430764630703983, 33.98560821371418, 2.9524683011979533, 69.34992427843227, 60.544590604607905, 51.64393601783645, 33.

,x,segment,unused_id,prediction
0,31.0,A,101,93.056448
1,32.0,B,102,101.013463
2,33.0,C,103,96.061544


### Inspect the preprocessed train/test frames

`get_fitted_split()` uses a throwaway preprocessing chain. It does not replace
the fitted preprocessing attached to `pipeline`.

The returned X frames are **already transformed**. They are suitable for a raw
estimator, not for `pipeline.predict()`, which would transform them again.

In [5]:
X_train, y_train, X_test, y_test = pipeline.get_fitted_split(
    data,
    target_column="target",
)
_require(len(X_train) + len(X_test) == len(data), "Splitting must preserve all input rows")
_require("unused_id" not in X_train.columns, "The configured identifier column must be dropped")
_require("target" not in X_train.columns, "The target must not remain among training features")
_require("x_double" in X_train.columns, "The fixed arithmetic feature must be generated")
np.testing.assert_allclose(X_train["x"].mean(), 0.0, atol=1e-12)

display(
    pd.DataFrame(
        {
            "partition": ["train", "test"],
            "rows": [len(X_train), len(X_test)],
            "scaled_x_mean": [X_train["x"].mean(), X_test["x"].mean()],
        }
    )
)
display(X_train.head())
np.testing.assert_allclose(
    np.asarray(pipeline.predict(new_rows)).reshape(-1),
    predictions,
)

,partition,rows,scaled_x_mean
0,train,45,0.000000
1,test,15,0.231253


,x,x_double,segment_A,segment_B,segment_C
17,-0.707362,-0.707362,0,0,1
8,-1.258288,-1.258288,0,0,1
6,-1.380716,-1.380716,1,0,0
40,0.700560,0.700560,0,1,0
4,-1.503144,-1.503144,0,1,0


## 4. Deliberately move the encoder before the split

This encoder learns a **feature vocabulary**, not just target labels. Moving it
before the split exposes it to held-out categories.

`leaking_pipeline` is just a descriptive Python variable name, not a special
class or an execution mode.

In [6]:
leaking_config = deepcopy(safe_config)
leaking_config["preprocessing"] = deepcopy(
    [
        feature_step,
        drop_step,
        encoding_step,
        split_step,
        target_step,
        scale_step,
    ]
)
leaking_pipeline = SkyulfPipeline(leaking_config)

try:
    leaking_pipeline.fit(data, target_column="target")
except ValueError as error:
    _require("Data leakage risk" in str(error), "The unsafe order must fail at the leakage gate")
    print(error)
else:
    raise AssertionError("Expected default raise mode to reject this order")

_require(not leaking_pipeline.is_fitted(), "The gate must run before learning")

try:
    leaking_pipeline.get_fitted_split(data, target_column="target")
except ValueError as error:
    _require("Data leakage risk" in str(error), "The unsafe order must fail at the leakage gate")
    print("Split extraction also rejects this unsafe order.")
else:
    raise AssertionError("get_fitted_split must not bypass the gate")

Data leakage risk:
Step 2 ('OneHotEncoder') is configured before the train/test split (step 3, 'TrainTestSplitter') and fits its statistics on the full dataset including the test set - move it after the splitter.
Split extraction also rejects this unsafe order.


### warn: continue, but explicitly report the risk

The config-only diagnostic returns messages; it does not fit or log them itself.
`fit(..., on_leakage="warn")` logs the messages and then continues.

This is a compatibility escape hatch, **not a safe solution**. Prefer moving the
encoder after the splitter. Do not present the following fit as leakage-free.

In [7]:
warning_messages = validate_leakage_safety(leaking_config, on_leakage="warn")
_require(bool(warning_messages), "The unsafe configuration must produce warning diagnostics")
display(warning_messages)

warning_pipeline = SkyulfPipeline(deepcopy(leaking_config))
warning_pipeline.fit(data, target_column="target", on_leakage="warn")
_require(warning_pipeline.is_fitted(), "Warning mode must permit the explicitly requested fit")
print("Fitted with an explicitly accepted leakage risk.")

["Step 2 ('OneHotEncoder') is configured before the train/test split (step 3, 'TrainTestSplitter') and fits its statistics on the full dataset including the test set - move it after the splitter."]

Fitted with an explicitly accepted leakage risk.


### ignore: continue without leakage warnings

This suppresses leakage diagnostics, not unrelated validation errors or logging.
The unsafe preprocessing order remains unsafe.

In [8]:
_require(
    validate_leakage_safety(leaking_config, on_leakage="ignore") == [],
    "Ignore mode must suppress leakage diagnostics",
)
ignored_pipeline = SkyulfPipeline(deepcopy(leaking_config))
ignored_pipeline.fit(data, target_column="target", on_leakage="ignore")
_require(ignored_pipeline.is_fitted(), "Ignore mode must permit the explicitly requested fit")

display(
    pd.DataFrame(
        [
            {"mode": "raise", "fit_allowed": False, "leakage_signal": "ValueError"},
            {"mode": "warn", "fit_allowed": True, "leakage_signal": "warning log"},
            {"mode": "ignore", "fit_allowed": True, "leakage_signal": "suppressed"},
        ]
    )
)

,mode,fit_allowed,leakage_signal
0,raise,False,ValueError
1,warn,True,warning log
2,ignore,True,suppressed


## 5. An external SplitDataset provides the boundary

Remove the configured train/test splitter when supplying raw, already-separated
partitions yourself. Keep the remaining feature-generation, dropping, encoding,
and scaling steps.

Both train and test must use the **training** mean and standard deviation.
The numerical checks below detect fitting the scaler on the combined data or
standardizing the held-out partition using its own statistics.

In [9]:
external_config = deepcopy(safe_config)
external_config["preprocessing"] = [
    step for step in external_config["preprocessing"] if step["transformer"] != "TrainTestSplitter"
]
external_data = SplitDataset(
    train=data.iloc[:45].copy(),
    test=data.iloc[45:].copy(),
)
external_pipeline = SkyulfPipeline(deepcopy(external_config))
external_pipeline.fit(external_data, target_column="target")

external_X_train, _, external_X_test, _ = external_pipeline.get_fitted_split(
    external_data,
    target_column="target",
)
if not isinstance(external_data.train, pd.DataFrame):
    raise RuntimeError("This example requires a pandas training partition")
if not isinstance(external_data.test, pd.DataFrame):
    raise RuntimeError("This example requires a pandas test partition")
train_mean = external_data.train["x"].mean()
train_std = external_data.train["x"].std(ddof=0)
np.testing.assert_allclose(
    external_X_train["x"],
    (external_data.train["x"] - train_mean) / train_std,
)
np.testing.assert_allclose(
    external_X_test["x"],
    (external_data.test["x"] - train_mean) / train_std,
)
display(
    pd.DataFrame(
        {
            "raw_test_x": external_data.test["x"],
            "actual_scaled_x": external_X_test["x"].to_numpy(),
            "expected_using_train_statistics": ((external_data.test["x"] - train_mean) / train_std),
        }
    )
)

,raw_test_x,actual_scaled_x,expected_using_train_statistics
45,23.118644,1.770978,1.770978
46,23.610169,1.847977,1.847977
47,24.101695,1.924976,1.924976
48,24.593220,2.001975,2.001975
49,25.084746,2.078974,2.078974
50,25.576271,2.155973,2.155973
51,26.067797,2.232972,2.232972
52,26.559322,2.309972,2.309972
53,27.050847,2.386971,2.386971
54,27.542373,2.463970,2.463970


### Why a config-only check still says "no split"

The diagnostic sees a config, not the external dataset passed to `fit()`.
With no configured splitter, `raise` and `warn` return an advisory; neither
raises a definite-violation error. `ignore` returns no messages.

Passing an unsplit DataFrame to such a pipeline is still allowed with an
advisory. It is **not** proof of safe evaluation. Supplying a proper
`SplitDataset`, as above, provides the missing boundary.

In [10]:
modes: tuple[OnLeakage, ...] = ("raise", "warn", "ignore")
diagnostics = []
for mode in modes:
    messages = validate_leakage_safety(external_config, on_leakage=mode)
    _require(
        bool(messages) == (mode != "ignore"), "Only ignore mode may suppress the no-split advisory"
    )
    diagnostics.append({"mode": mode, "messages": messages})
display(pd.DataFrame(diagnostics))

,mode,messages
0,raise,[No train/test split is defined in this pipeli...
1,warn,[No train/test split is defined in this pipeli...
2,ignore,[]


## 6. Core-only cross-validation: a fresh pipeline per fold

A train/test split and per-fold refitting solve different problems. During CV,
do not fit the encoder/scaler on all rows and only then split the transformed
frame into folds.

Instead, split **raw rows** and create a fresh pipeline inside each iteration.
The `SplitDataset` supplied to that instance makes its preprocessing fit on
that fold's training rows only. This explicit core loop does not require the
backend's graph-to-fold adapter.

The results below are illustrative CV measurements, not model-selection results
from an independent final test set. Reserve an additional untouched test set
when using CV to choose a production model.

In [11]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)
fold_results = []
for fold_number, (train_indices, validation_indices) in enumerate(cv.split(data), start=1):
    fold_data = SplitDataset(
        train=data.iloc[train_indices].copy(),
        test=data.iloc[validation_indices].copy(),
    )
    # Never reuse a previously fitted pipeline across folds.
    fold_pipeline = SkyulfPipeline(deepcopy(external_config))
    fold_pipeline.fit(fold_data, target_column="target")
    raw_validation_X = fold_data.test.drop(columns=["target"])
    fold_predictions = np.asarray(fold_pipeline.predict(raw_validation_X)).reshape(-1)
    mse = mean_squared_error(fold_data.test["target"], fold_predictions)
    fold_results.append(
        {
            "fold": fold_number,
            "train_rows": len(train_indices),
            "validation_rows": len(validation_indices),
            "mse": float(mse),
        }
    )

cv_report = pd.DataFrame(fold_results)
_require(len(cv_report) == 3, "Cross-validation must report all three folds")
_require(
    bool(np.isfinite(cv_report["mse"]).all()), "Each fold must produce a finite validation error"
)
display(cv_report)
print("Mean fold MSE:", cv_report["mse"].mean())

,fold,train_rows,validation_rows,mse
0,1,40,20,0.034287
1,2,40,20,0.023159
2,3,40,20,0.022004


Mean fold MSE: 0.0264831787108028


### 6.1 Native core tuning: automatic refit per candidate fold

The same preprocessing is reused as a configuration, not as a pre-fitted transform.
Tuning settings are flat modeling keys. With no validation partition, this runs
two ridge candidates over three inner folds, then fits the final artifact on
the outer training partition. Predictions reuse that final artifact.


In [12]:
tuning_config = deepcopy(safe_config)
tuning_config["modeling"] = {
    "type": "hyperparameter_tuner",
    "base_model": {"type": "ridge_regression"},
    "strategy": "grid",
    "metric": "r2",
    "search_space": {"alpha": [0.1, 1.0]},
    "cv_folds": 3,
    "random_state": 42,
}
tuned_pipeline = SkyulfPipeline(tuning_config)
tuning_metrics = tuned_pipeline.fit(data, target_column="target")
_require(tuned_pipeline.is_fitted(), "Native tuning must produce a fitted pipeline")
_require(
    bool(tuning_metrics["preprocessing"]["steps"]),
    "Native tuning must retain preprocessing metrics",
)
tuned_predictions = np.asarray(tuned_pipeline.predict(new_rows)).reshape(-1)
_require(
    bool(np.isfinite(tuned_predictions).all()), "The tuned pipeline must produce finite predictions"
)
display(new_rows.assign(tuned_prediction=tuned_predictions))

,x,segment,unused_id,tuned_prediction
0,31.0,A,101,93.000837
1,32.0,B,102,100.925785
2,33.0,C,103,96.022125


## 7. Core and backend: boundaries and supported graphs

The separate guide is `docs/user_guide/leakage_core_backend.md`; the complete
node inventory is `docs/user_guide/preprocessing_leakage_audit.md`.
These SVG diagrams are embedded attachments. They do not require an internet
connection or a Mermaid plugin; editable Mermaid sources accompany the guide.

### Core: learn on training rows, reuse on held-out rows



Core uses an ordered preprocessing list or an external `SplitDataset`.
Section 6 creates a new pipeline per raw-data fold; section 6.1 demonstrates
native core tuning with automatic per-fold preprocessing refits.

### Backend: the simplest supported chain



Yeo-Johnson and Box-Cox belong after the actual train/test row split.
Feature-Target Split separates X/y but creates no held-out row boundary.

### Your screenshot: fixed branches merging at the splitter



This shape now supports fold refitting when the branches share one loader,
contain only fixed operations, and the downstream chain is reconstructable.
The backend retains the merged raw input at the first row split.

Replacing the fixed feature formula with Yeo-Johnson makes it unsafe before
the splitter and the default policy rejects it. FeatureGeneration group
aggregates also learn: they now fit a training lookup and reuse it for held-out
rows, with unseen groups producing missing values.

| Outcome | Meaning |
|---|---|
| Submission rejected with HTTP 400 | Admission found a definite violation before creating jobs |
| Fold refit enabled | Candidate preprocessing is fitted inside each fold |
| Unsupported learned path with raise | Execution stops instead of publishing contaminated CV scores |
| Unsupported path with explicit warn/ignore | Fallback is permitted, not repaired |

The guard cannot prove that a feature excludes future information, target-derived
inputs, or preprocessing already performed outside the pipeline. See the
per-node audit for temporal features, compatibility, and test boundaries.


### 7.1 Find placement rules and canvas feedback

The [preprocessing placement guide](../../docs/user_guide/preprocessing_placement.md)
lists every registration as fixed, conditional, after-split, or time-sensitive.
Use it when changing a node's settings: Yeo-Johnson and log share a node but
have different fitting requirements.

In the canvas, open **How pipelines work > Preprocessing & Leakage** and search
by node name. Definite ordering violations appear on the offending node and
connection; select their warning symbol for the explanation and guide link.
Amber markers are advisories, not blocking violations. Backend errors that
cannot be located precisely use a compact dismissible notice rather than
opening Preview Results. Ordinary preview errors still use that panel.

These are canvas presentation changes. The standalone core examples above
continue to use `raise`, `warn`, and `ignore` through the Python API.
